# Task 1: News Topic Classifier Using BERT
**Objective:**<br>
Fine-tune a transformer model (e.g., BERT) to classify news headlines into topic categories.<br>
**Dataset:**<br>
AG News Dataset (Available on Hugging Face Datasets)<br>
**Instructions:**<br>
● Tokenize and preprocess the dataset<br>
● Fine-tune the bert-base-uncased model using Hugging Face Transformers<br>
● Evaluate the model using accuracy and F1-score<br>
● Deploy the model using Streamlit or Gradio for live interaction<br>
**Skills Gained:**<br>
● NLP using Transformers<br>
● Transfer learning & fine-tuning<br>
● Evaluation metrics for text classification<br>
● Lightweight model deployment <br>

# Install libraries

In [ ]:
!pip install transformers
!pip install datasets
!pip install torch
!pip install streamlit
!pip install sklearn

# Step 1 : Load data set

In [13]:
from datasets import load_dataset

In [14]:
dataset = load_dataset("SUBHANmm/Smaller_AG_News_Dataset")

In [15]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 5900
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 4000
    })
})

In [16]:
data_train = load_dataset("SUBHANmm/Smaller_AG_News_Dataset" , split="train")

In [17]:
data_train

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 5900
})

In [18]:
data_train[1]

{'text': "BBC reporters' log BBC correspondents record events in the Middle East and their thoughts as the funeral of the Palestinian leader Yasser Arafat takes place.",
 'label': 0,
 '__index_level_0__': 88708}

In [19]:
data_test = load_dataset("SUBHANmm/Smaller_AG_News_Dataset" , split="test")

In [20]:
data_test

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 4000
})

In [21]:
data_test[:5]

{'text': ['Peruvian Maoist trial thrown into chaos The first hearing in the re-trial of former leaders of Peru #39;s Shining Path guerrilla group has ended in chaos. The judge suspended the hearing after the group #39;s founder, Abimael Guzman, and his 15 co-defendants ',
  'Running may have defined the body Next time you drive past a jogger on the street, give her a honk and a wave - she #39;s honing the skill that helped define the human body, according to a study by researchers from the University of Utah and Harvard.',
  'Report: CARE Hostage Faces Transfer To Al-Qaida BAGHDAD, Iraq -- The kidnappers of aid worker Margaret Hassan threatened to turn her over to an al-Qaida affiliated group within 48 hours if the British government refuses to pull its troops from Iraq, Al-Jazeera television reported Tuesday.',
  'Intel Cancels Plan to Enter Digital TV Chip Market  SAN FRANCISCO (Reuters) - Intel Corp. &lt;A HREF="http://www.reuters.co.uk/financeQuoteLookup.jhtml?ticker=INTC.O qtype=s

In [10]:
!git lfs install

Git LFS initialized.


In [11]:
!git clone https://huggingface.co/datasets/SUBHANmm/Smaller_AG_News_Dataset

fatal: destination path 'Smaller_AG_News_Dataset' already exists and is not an empty directory.


In [22]:
dataset = load_dataset("./Smaller_AG_News_Dataset")

In [23]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 5900
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 4000
    })
})

In [24]:
traing_data = load_dataset("./Smaller_AG_News_Dataset" , split="train")

In [25]:
traing_data

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 5900
})

In [26]:
traing_data[0]

{'text': 'Explosion Rocks Baghdad Neighborhood BAGHDAD, Iraq, August 24 -- A car bomb exploded near the gate of a US-funded Iraqi television network in Baghdad on Tuesday, killing at least two people and wounding two others, authorities and witnesses said.',
 'label': 0,
 '__index_level_0__': 8897}

In [27]:
testing_data = load_dataset("./Smaller_AG_News_Dataset" , split="test")

In [28]:
testing_data

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 4000
})

In [29]:
testing_data[1]

{'text': 'Running may have defined the body Next time you drive past a jogger on the street, give her a honk and a wave - she #39;s honing the skill that helped define the human body, according to a study by researchers from the University of Utah and Harvard.',
 'label': 3,
 '__index_level_0__': None}

# Step 2 :  Tokenize and preprocess the dataset

In [30]:
# Remove Unnecessary columns

dataset = dataset.remove_columns(['__index_level_0__'])

In [31]:
# Checks Labels
import numpy as np

print(np.unique(dataset["train"]["label"]))

[0 1 2 3]


In [32]:
# Text check
print(dataset["train"][0])

{'text': 'Explosion Rocks Baghdad Neighborhood BAGHDAD, Iraq, August 24 -- A car bomb exploded near the gate of a US-funded Iraqi television network in Baghdad on Tuesday, killing at least two people and wounding two others, authorities and witnesses said.', 'label': 0}


### Tokenize

In [33]:
from transformers import BertTokenizer

In [34]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [52]:
# tokenize function
def tokenize(data):
    return tokenizer(
        data["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

### Split data

In [53]:
tokenizer_train = dataset["train"].map(tokenize)

Map:   0%|          | 0/5900 [00:00<?, ? examples/s]

In [54]:
tokenizer_train

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5900
})

In [55]:
tokenizer_test = dataset["test"].map(tokenize)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [56]:
tokenizer_test

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})

# Step 3 : Fine-tune the bert-base-uncased model using Hugging Face Transformers

###  BERT Model load

In [40]:
from transformers import BertForSequenceClassification

In [41]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 4
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Training Arguments

In [42]:
from transformers import TrainingArguments

In [ ]:
!pip install transformers[torch] --upgrade

In [ ]:
!pip install accelerate --upgrade

In [43]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch"
)

In [44]:
print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

### Train Model 

In [47]:
# Accuracy Function
from sklearn.metrics import accuracy_score  , f1_score

In [48]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')

    return {
        "accuracy":acc,
        "f1":f1
    } 

In [49]:
# make model trainer set

from transformers import Trainer

In [57]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenizer_train,    
    eval_dataset=tokenizer_test,     
    compute_metrics=compute_metrics 
)

In [58]:
# train model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.344174,0.893000,0.893137
2,0.539594,0.322925,0.908250,0.908549
3,0.128283,0.385248,0.917250,0.917356


C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=1107, training_loss=0.30742709401931007, metrics={'train_runtime': 19724.7071, 'train_samples_per_second': 0.897, 'train_steps_per_second': 0.056, 'total_flos': 1164287326924800.0, 'train_loss': 0.30742709401931007, 'epoch': 3.0})

### Save Model

In [59]:
model.save_pretrained('./my_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### save Tokenizer

In [60]:
tokenizer.save_pretrained('./my_model')

('./my_model\\tokenizer_config.json', './my_model\\tokenizer.json')